Word pattern matching. very tractable

hedge rewriting, sequence variables

mathematica has a_ a__ a___
https://reference.wolfram.com/language/ref/BlankSequence.html

https://link.springer.com/chapter/10.1007/3-540-45470-5_26
Temur Kutsia
https://www3.risc.jku.at/people/tkutsia/papers/Kutsia_JSC_2007.pdf


syntax parse. x... patterns

https://ui.adsabs.harvard.edu/abs/arXiv:1503.00336 clp hedge
https://www.jaist.ac.jp/flops2014/dundua.pdf slides

Context variables too. Hmm. Hedges are term forests? Got for category idea?

A-matching, not a AC. But if there is not really such a thing as flattening that seems fine too. Just multi-arity.

https://docs.racket-lang.org/guide/match.html Hmm. But also one could have foo(X)*  as sequence, such that all members of the sequence have to match that? But then X is individually bound (probably). Hmm.
https://docs.racket-lang.org/syntax/stxparse-patterns.html


arity constraints enable len(X) == 1 -> X is a character
len(X) > 0
len(X) < 17
interval constraints


Hmm. Sequence patterns are interesting for code blocks


DAGs as containers. A dag could represent the set of all sequentializations of the dag.
Then a sequence pattern could just return those that match, but it is more efficient to collect some of these up in a factored form
Is the dependence a natural property of the item? It isn't so clear.


a X* b

concat patterns. concat(X,concat(a, ))  pick a splitting point. That is kind of interesting too. Concat patterns vs sequence patterns. I don't think concat is persay that natural feeling in this context.  

The string egraph




In [13]:
from dataclasses import dataclass
@dataclass
class Var:
    name : str
type Subst = dict[str, tuple[object, ...]]
def pmatch_rec(pattern : tuple[object | Var, ...], seq: tuple, subst0) -> list[Subst]:
    print(pattern, seq, subst0)
    if len(pattern) == 0:
        if len(seq) == 0:
            return [subst0] #succeed 
        else:
            return [] # fail if pattern is empty but sequence is not
    else:
        match pattern[0]:
            case Var(v):
                if v in subst0:
                    k = subst0[v]
                    if len(k) > len(seq) or seq[:len(k)] != k:
                        return [] # fail if the sequence does not match the substitution
                    else:
                        return pmatch_rec(pattern[1:], seq[len(k):], subst0)
                results = []
                for i in range(len(seq)+1): # 0 or all. Could also allow nonzero patterns
                    subst = subst0.copy()
                    subst[v] = seq[:i]
                    results.extend(pmatch_rec(pattern[1:], seq[i:], subst))
                return results
            case p:
                if len(seq) == 0:
                    return [] # fail
                elif seq[0] == p:
                    return pmatch_rec(pattern[1:], seq[1:], subst0)
                else:
                    return [] 
# could also count variables occurence and prune earlier if known length exceeds.
# It also feels like maybe we'd want to traverse in two directions to start with a constant. But whatever.

def pmatch(pattern : tuple[object | Var, ...], seq: tuple) -> list[Subst]:
    return pmatch_rec(pattern, seq, {})

print(pmatch([Var("x"), Var("y")], ("a", "b")))
print(pmatch([Var("x"), Var("x")], ("a", "a")))

[Var(name='x'), Var(name='y')] ('a', 'b') {}
[Var(name='y')] ('a', 'b') {'x': ()}
[] ('a', 'b') {'x': (), 'y': ()}
[] ('b',) {'x': (), 'y': ('a',)}
[] () {'x': (), 'y': ('a', 'b')}
[Var(name='y')] ('b',) {'x': ('a',)}
[] ('b',) {'x': ('a',), 'y': ()}
[] () {'x': ('a',), 'y': ('b',)}
[Var(name='y')] () {'x': ('a', 'b')}
[] () {'x': ('a', 'b'), 'y': ()}
[{'x': (), 'y': ('a', 'b')}, {'x': ('a',), 'y': ('b',)}, {'x': ('a', 'b'), 'y': ()}]
[Var(name='x'), Var(name='x')] ('a', 'a') {}
[Var(name='x')] ('a', 'a') {'x': ()}
[] ('a', 'a') {'x': ()}
[Var(name='x')] ('a',) {'x': ('a',)}
[] () {'x': ('a',)}
[Var(name='x')] () {'x': ('a', 'a')}
[{'x': ('a',)}]


# 2026-09

Listening to aaron stump's podcast https://podcasts.apple.com/us/podcast/solving-quadratic-word-equations/id1493036757?i=1000782783267 " My source is Chapter 12 of "Algebraic Combinatorics on Words" by Lothaire."

You want to know if you've revissted a state. Tabling.
But taking a set of equations and hash tabling them is actually annoying



Word equations is exists. Solving over the free theory. Makanin
KB s forall. Finite presentations

Nielson transformation
Pete's paper


A trivially solvable subclass gives us a String union find


Self symmettries are non trivial and interesting. Cyclic constraints? w^n

aX = Xb

fXf --> f(X(f(...)))  unary context variables?


Queue solvers. push and pop tell us about the front and back of the queue, but maybe don't tell use about the interior. pop(a,push(b,Q1)) = Q2. Something like that. The push and pop commute with respect to each other, but not amongst themselves
z3-str
noodler
string solvers
STeP





In [ ]:
from dataclasses import dataclass, field
type Id = int
type FatId = tuple[str, Id, str]

"""
Overloading
@dataclass(frozen=True, slots=True)
class FatId:
    pre : str
    idx : Id
    post : str

    def __add__(sefl)
"""
@dataclass
class StringUF:
    parents : list[FatId] = field(default_factory=list)
    # symmettries

    def makeset(self):
        id = ("", len(self.parents), "")
        self.parents.append(id)
        return id
    def find(self, x : FatId) -> FatId:
        pre, idx, post = x
        while True:
            (pre1, idx1, post1) = self.parents[idx]
            if idx1 == idx:
                assert pre1 == "" and post1 == ""
                return (pre, idx, post)
            else:
                pre, idx, post = pre + pre1, idx1, post1 + post
    def union(self, x : FatId, y : FatId):
        (prex, xid, postx), (prey, yid, posty) = self.find(x), self.find(y)
        # peel off commono prefix
        if xid == yid: # redundant
            assert prex == prey and postx == posty # inconsistent
            return
        # peel off common suffix
        Npre = min(len(prex), len(prey))
        assert prex[:Npre] == prey[:Npre]
        prex, prey = prex[Npre:], prey[Npre:]

        Npost = min(len(postx), len(posty))
        assert post[:]

        for n in range(min(len(prex), len(prey)))
            if prex[n] != prey[n]:
                break
        prex, prey = prex[n:], prey[n:]
        for n, (x,y) in enumerate(zip(reversed(postx),reversed(posty))):
            if x != y:
                break
        postx, posty = postx[:-n-1], posty[:-n-1]
        
        if postx == "" and prex == "":
            self.parents[xid] = (prey, yid, posty)
        elif posty == "" and prey == "":
            self.parents[yid] = (prex, xid, postx)
        else:
            _,zid,_ = self.makeset()
            print(prex, xid, postx, "y", prey, yid, posty)
            assert prex == "" or prey == ""
            assert postx == "" or posty == ""
            self.parents[xid] = (prey, zid ,posty)
            self.parents[yid] = (prex , zid,  postx)
    def is_eq(self, x : FatId, y : FatId) -> bool:
        return self.find(x) == self.find(y)



        # solves equations of the form cccXccc = cccYccc
        # strip off common prefix and suffix

def app(pre, fid : FatId, post=""):
    return (pre + fid[0], fid[1], fid[2] + post) 

uf = StringUF()
x,y,z = [uf.makeset() for _ in range(3)]
uf.union(app("aa", x), app("aaa", y))




Hash consing knuth bendix such that every equations is of the form 
ab -> aaaaaaa

Makes overlap easier (not that it was that hard)

Slava monoid.net
Maybe I should update my old 

https://github.com/tekknolagi/knuth-bendix/blob/trunk/main.rb
https://www.philipzucker.com/string_knuth/ 
https://github.com/philzook58/eggmt/blob/main/eggmt/string_kb.py




# 2026-01

https://link.springer.com/chapter/10.1007/3-540-61208-4_16 T-string unification: Unifying prefixes in non-classical proof methods
Somehow string unification is relevant to nonclassical tableau proving. That's interesting.
L. Wallen. Automated deduction in nonclassical logic. MIT Press, 1990.


String equations
string unification
word problem
uniform word problem

https://skos.ii.uni.wroc.pl/pluginfile.php/73431/mod_resource/content/9/notes.pdf notes on word equations solving

https://youtu.be/MLwZrU6dOwE?si=TuKdKXKCj65xsYXS https://icetcs.github.io/frocos-itp-tableaux25/slides/tblfro/tableaux7-eisenhofer.pdf https://doi.org/10.1007/978-3-032-06085-3_7 On Solving String Equations via Powers and Parikh Images
https://icetcs.github.io/frocos-itp-tableaux25/recordings.html 

"
- Automatons (z3-noodler, Ostrich, Sloth, TRAU, . . . )
- SAT (Woorpje, nfa2sat)
- Unwinding/model repair (z3, cvc5, S3, z3Str3, . . . )
- Nielsen transformation (more or less in all solvers)
Termination only on “common-fragments”
quadratic, 2-variable, straight-line, chain-free
"

Nielson - from free grou thoery. Non temrimating in general https://en.wikipedia.org/wiki/Nielsen_transformation . Combinatorial group theory. 
parikh ~ character count. Can show some eqwuations can't be solved

Even if nielson happens inside, I could do it outseide.



word constraints
string constraints

word equations https://en.wikipedia.org/wiki/Word_equation
https://en.wikipedia.org/wiki/Combinatorics_on_words

https://drops.dagstuhl.de/storage/00lipics/lipics-vol152-csl2020/LIPIcs.CSL.2020.3/LIPIcs.CSL.2020.3.pdf solving word equatioms by recompressions 2020
also works on traces (partially commutative monoids?)
bubble sort for canonization.

https://github.com/remora-lang/remora/blob/main/src/Symbolic.hs#L65-L88

https://github.com/jrslepak/makanin-algo


https://homepage.divms.uiowa.edu/~ajreynol/pres-tpss2021.pdf cvc5 presentation summary of string solvers

https://assets.amazon.science/be/e6/c0265a14477ca4558784008404ea/solving-string-constraints-using-sat.pdf solving strings using sat 2023 aws

z3-noodler

cvc5

ostrich2

libmata


necklaces and rings.
cycle rewriting. zantema https://arxiv.org/abs/1609.07065 Termination of Cycle Rewriting by Transformation and Matrix Interpretation
https://cofault.com/aodt.html#aodt
Bag(Ring(x)) = e^Ring(x) = List(x) = 1/(1-x)
https://en.wikipedia.org/wiki/Monoid_factorisation#Chen%E2%80%93Fox%E2%80%93Lyndon_theorem   Chen–Fox–Lyndon theorem
Lyndon decomposition of monoid
https://en.wikipedia.org/wiki/Lyndon_word
A necklace as something else in the list, set, multiset hierarchy. You can't really associated

Nielson steps

```
au = bv
------
⊥

au = ε
------
⊥

wu = wv
------
u = v

xu = ε
-------
(xu = ε)[x/ε]

xu = av
------
(xu = av)[x/ε] (xu = av)[x/ax′]

xu = yv
------
(xu = yv)[x/y] (xu = yv)[x/yx′] (xu = yv)[y/xy′]
```

https://www.ccs.neu.edu/home/pete/pub/fmcad-2021.pdf Mathematical Programming Modulo Strings. Pete 2021

In [1]:
from dataclasses import dataclass
from typing import NamedTuple

class Var(NamedTuple):
    name : str

def subst(xs : list[object | Var], n : Var, s):
    res = []
    for x in xs:
        if n == x:
            res.extend(s)
        else:
            res.append(x)
    return res

counter = 0
def freshvar():
    global counter
    counter += 1
    return Var(f"_{counter}")


def subst_pair(pair : tuple[list[object | Var], list[object | Var]], n : Var, s):
    x,y = pair
    return (subst(x, n, s), subst(y, n, s))
from collections import deque

def nielson(x : list[object | Var], y : list[object | Var]):
    # mayb apply the nielson rules to front and back as is useful
    q = deque()
    subst = {}
    q.append((subst, ((x,y))))
    seen = set()
    n = 0
    while q:
        n += 1
        if n > 10:
            return
        (subst, (x,y)) = q.popleft()
        print("Nielson step:", x, y, subst)
        while x and y:
            a,b = x.pop(), y.pop()
            if a == b:
                continue
            else:
                # not equal. Do the junk
                # subst_pair copies btw
                x.append(a)
                y.append(b)
                p = (x,y)
                if not isinstance(a, Var) and not isinstance(b, Var):
                   pass # fail
                elif isinstance(a, Var) and isinstance(b, Var):
                    # both vars
                    z = freshvar()
                    q.append(({a : [b], **subst}, subst_pair(p, a, b ))) # assume equal
                    q.append(({b : [z,a] , **subst}, subst_pair(p, b, [z,a]) )) # a is prefix of b
                    q.append(({a : [z,b], **subst}, subst_pair(p, a, [z,b]) )) # b is prefix of a
                else:
                    if isinstance(b, Var):
                        a,b = b,a
                    # a is var, b is not
                    q.append(({a : [], **subst}, subst_pair((x,y), a, []))) # a is empty
                    #q.append(({a : [b], **subst}, subst_pair((x,y), a, [b]))) # a is empty
                    z = freshvar()
                    q.append(({a : [z,b], **subst}, subst_pair((x,y), a, [z,b]))) # b is prefix of a
                break
        else:
            if len(x) == 0 and len(y) == 0:
                yield subst
            elif len(y) == 0:
                if all(isinstance(v, Var) for v in x):
                    yield {**subst, **{v : [] for v in x}}
            elif len(x) == 0:
                if all(isinstance(v, Var) for v in y):
                    yield {**subst, **{v : [] for v in y}}

def canon_subst(subst, x):
    res = []
    for a in x:
        if isinstance(a, Var) and a in subst:
            subst[a] = Var(len(subst))
        else:
            res.append(a)
    return tuple(res)

def canon(p):
    x,y = p
    subst = {}
    x1 = canon_subst(subst, x)
    y1 = canon_subst(subst, y)
    return (x1,y1)


def compress(subst):
    res = {}
    for k,v in subst.items():
        s = []
        i = 0
        todo = list(reversed(v))
        while todo:
            x = todo.pop()
            if isinstance(x, Var) and x in subst:
                todo.extend(subst[x])
            else:
                s.append(x)
        res[k] = s
    return res

def cnielson(x : list[object | Var], y : list[object | Var]):
    for subst in nielson(x,y):
        yield {k:v for k,v in compress(subst).items() if not k.name.startswith("_")}

x,y,z = map(Var, "xyz")
assert list(cnielson([1,2], [1,2])) == [{}]
assert list(cnielson([1,2], [1,2,3])) == []
list(cnielson([x,2], [1,2]))
#assert list(nielson([x,2], [1,x])) == [] # doesn't terminate
assert list(cnielson([x,2], [1,2,3])) == []
list(cnielson([x,y], [1,2]))
list(cnielson([x,x], [1,2]))
list(cnielson([x,2], [2,x]))
#list(nielson([1,x], [1,2,y]))

Nielson step: [1, 2] [1, 2] {}
Nielson step: [1, 2] [1, 2, 3] {}
Nielson step: [Var(name='x'), 2] [1, 2] {}
Nielson step: [] [1] {Var(name='x'): []}
Nielson step: [Var(name='_1'), 1] [1] {Var(name='x'): [Var(name='_1'), 1]}
Nielson step: [Var(name='x'), 2] [1, 2, 3] {}
Nielson step: [Var(name='x'), Var(name='y')] [1, 2] {}
Nielson step: [Var(name='x')] [1, 2] {Var(name='y'): []}
Nielson step: [Var(name='x'), Var(name='_2'), 2] [1, 2] {Var(name='y'): [Var(name='_2'), 2]}
Nielson step: [] [1, 2] {Var(name='x'): [], Var(name='y'): []}
Nielson step: [Var(name='_3'), 2] [1, 2] {Var(name='x'): [Var(name='_3'), 2], Var(name='y'): []}
Nielson step: [Var(name='x')] [1] {Var(name='_2'): [], Var(name='y'): [Var(name='_2'), 2]}
Nielson step: [Var(name='x'), Var(name='_4'), 1] [1] {Var(name='_2'): [Var(name='_4'), 1], Var(name='y'): [Var(name='_2'), 2]}
Nielson step: [] [1] {Var(name='_3'): [], Var(name='x'): [Var(name='_3'), 2], Var(name='y'): []}
Nielson step: [Var(name='_5'), 1] [1] {Var(name='_

[{Var(name='x'): []},
 {Var(name='x'): [2]},
 {Var(name='x'): [2, 2]},
 {Var(name='x'): [2, 2, 2]},
 {Var(name='x'): [2, 2, 2, 2]}]